In [ ]:
#@title Copyright 2026 Google LLC
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Weather forecasting and storm risk nowcasting with `Earth Engine`, BigQuery ML, and Gemini

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/python-docs-samples/blob/feat/modernize-ppai-weather-forecasting/people-and-planet-ai/weather-forecasting/README.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fpython-docs-samples%2Ffeat%2Fmodernize-ppai-weather-forecasting%2Fpeople-and-planet-ai%2Fweather-forecasting%2FREADME.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/python-docs-samples/feat/modernize-ppai-weather-forecasting/people-and-planet-ai/weather-forecasting/README.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Vertex AI logo"><br> Open in Vertex AI Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/python-docs-samples/blob/feat/modernize-ppai-weather-forecasting/people-and-planet-ai/weather-forecasting/README.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/generative-ai/logos/GitHub_Invertocat_Dark.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

## Overview

Weather forecasting combines three distinct time horizons—and you do not need a background in meteorology to work with them on Google Cloud:

1. **Observations (Current conditions)**: Satellites in orbit measure cloud-top temperatures and falling rain, while digital elevation maps capture mountain ranges and river valleys.
2. **Nowcasting (Next 1 to 6 hours)**: Foundation time-series models (**TimesFM 2.5** in BigQuery ML `AI.FORECAST`) project recent hourly rainfall trends forward with uncertainty bounds, helping emergency responders anticipate flash floods before river gauges crest.
3. **Medium-range and ensemble forecasting (Next 3 to 15 days)**: Global numerical and AI weather models (**NOAA GFS** and **Google DeepMind WeatherNext 3**) simulate dozens of possible future atmospheres ("ensembles") to capture both the most likely forecast and worst-case extremes.

### Storm and regional case studies included

* **Hurricane Melissa (`2025-10-28T17:00:00Z`, Jamaica landfall — default)**: Category 5 landfall in southwestern Jamaica (`18.1N, 78.0W`), where spiral rainbands collide with the steep Blue Mountains (`2,256 m` peak elevation).
* **Hurricane Helene (`2024-09-27T06:00:00Z`, Southern Appalachians)**: Historic inland atmospheric moisture plume interacting with Blue Ridge mountain topography (`35.60N, 82.55W`, up to `2,037 m` elevation).
* **Live local weather (`San Francisco Bay Area, CA` or your own coordinates)**: Real-time West Coast satellite imagery (`GOES-18`), Bay Area topography (`MERIT DEM`), and a live 72-hour forecast (`NOAA GFS`).

### Objectives

* Retrieve and interpret multi-band Earth Engine satellite (`GOES-19`/`GOES-18`/`GOES-16`), precipitation (`GPM IMERG V07`), and elevation (`MERIT DEM`) rasters (`fetch_weather_patch.py`).
* Populate a BigQuery hourly storm time-series table using Apache Beam (`load_weather_time_series.py`).
* Execute zero-shot precipitation nowcasting in BigQuery ML with `AI.FORECAST` and `TimesFM 2.5` (`forecast_precipitation.py`).
* Generate a structured `StormRiskAssessment` report from satellite imagery and tabular forecasts using `gemini-3.8-flash` (`analyze_storm_risk.py`).
* Customize the pipeline for live local forecasts in the San Francisco Bay Area and explore 64-member probabilistic forecasts from **WeatherNext 3**.

### Costs

This tutorial uses billable components of Google Cloud:

* [BigQuery](https://cloud.google.com/bigquery/pricing) (storage and `AI.FORECAST` query processing)
* [Dataflow](https://cloud.google.com/dataflow/pricing) (if running the Apache Beam pipeline on `DataflowRunner`; defaults to local `DirectRunner` in this notebook)
* [Vertex AI / Gemini API](https://cloud.google.com/vertex-ai/generative-ai/pricing) (`gemini-3.8-flash` multimodal token usage)
* [Cloud Storage](https://cloud.google.com/storage/pricing) (temporary staging bucket for BigQuery file loads)

## Get started

### Clone the sample repository and install dependencies

Fetch the `people-and-planet-ai/weather-forecasting` sample scripts and install the required Python packages into the current notebook kernel (~30 seconds).

In [ ]:
import importlib
import os
import subprocess
import sys

REPO_URL = "https://github.com/GoogleCloudPlatform/python-docs-samples.git"
BRANCH = "feat/modernize-ppai-weather-forecasting"
SAMPLE_DIR = "people-and-planet-ai/weather-forecasting"

if not os.path.exists("fetch_weather_patch.py"):
    if not os.path.exists("python-docs-samples"):
        subprocess.run([
            "git", "clone", "--depth", "1", "--filter=blob:none",
            "--sparse", "--branch", BRANCH, REPO_URL
        ], check=True)
        subprocess.run([
            "git", "-C", "python-docs-samples", "sparse-checkout", "set", SAMPLE_DIR
        ], check=True)
    os.chdir(os.path.join("python-docs-samples", SAMPLE_DIR))

sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

Install the sample dependencies into the active Python environment using `%pip install`.

In [ ]:
%pip install --upgrade --quiet -r requirements.txt

### Authenticate your notebook environment

If you are running this notebook in Google Colab, authenticate with your Google Cloud user credentials. In Vertex AI Workbench or Colab Enterprise, Application Default Credentials (ADC) are preconfigured automatically.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Configure your Google Cloud project and storm preset

Set your `PROJECT_ID`, `BUCKET_NAME`, `BIGQUERY_DATASET`, and choose between `melissa_2025` (Hurricane Melissa, Oct 28, 2025, Jamaica) and `helene_2024` (Hurricane Helene, Sept 27, 2024, Southern Appalachians).

> **Tip for Earth Engine access**: To pull live satellite pixels from Google Earth Engine, make sure your project is registered (free for research, education, and developer testing) at [console.cloud.google.com/earth-engine/configuration](https://console.cloud.google.com/earth-engine/configuration).

In [ ]:
import logging
import os
import subprocess
import warnings

logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning, module=r"pandas\.core\.arrays\.masked")
os.environ["PYTHONWARNINGS"] = "ignore::UserWarning:pandas.core.arrays.masked"

PROJECT_ID = "example-project-id"  # @param {type:"string"}
LOCATION = "US"  # @param {type:"string"}
BUCKET_NAME = "example-weather-bucket"  # @param {type:"string"}
BIGQUERY_DATASET = "example_weather_forecasting"  # @param {type:"string"}
BIGQUERY_TABLE = "example_storm_observations"  # @param {type:"string"}
STORM_PRESET = "melissa_2025"  # @param ["melissa_2025", "helene_2024"]

if not PROJECT_ID or PROJECT_ID == "example-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT") or subprocess.check_output(
        ["gcloud", "config", "get-value", "project"], text=True
    ).strip()
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", LOCATION)
if not BUCKET_NAME or BUCKET_NAME == "example-weather-bucket":
    BUCKET_NAME = os.environ.get("BUCKET_NAME", f"{PROJECT_ID}-weather-forecasting")
BIGQUERY_DATASET = os.environ.get("BIGQUERY_DATASET", BIGQUERY_DATASET)
BIGQUERY_TABLE = os.environ.get("BIGQUERY_TABLE", BIGQUERY_TABLE)
STORM_PRESET = os.environ.get("STORM_PRESET", STORM_PRESET)

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
print(f"Project ID:       {PROJECT_ID}")
print(f"Staging Bucket:   gs://{BUCKET_NAME}")
print(f"BigQuery Dataset: {PROJECT_ID}.{BIGQUERY_DATASET}")
print(f"Storm Preset:     {STORM_PRESET}")

### Enable required Google Cloud APIs and create the staging bucket

Enable the BigQuery, Vertex AI (`aiplatform`), Earth Engine, and Dataflow APIs, and create a regional Cloud Storage bucket with soft-delete disabled for temporary BigQuery staging files (~30 seconds).

In [ ]:
!gcloud services enable bigquery.googleapis.com aiplatform.googleapis.com dataflow.googleapis.com earthengine.googleapis.com storage.googleapis.com --project={PROJECT_ID}

bucket_check = subprocess.run(
    ["gcloud", "storage", "buckets", "describe", f"gs://{BUCKET_NAME}", f"--project={PROJECT_ID}"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
if bucket_check.returncode != 0:
    !gcloud storage buckets create gs://{BUCKET_NAME} --project={PROJECT_ID} --location={LOCATION} --soft-delete-duration=0s
else:
    !gcloud storage buckets update gs://{BUCKET_NAME} --clear-soft-delete

## 1. Retrieve and visualize satellite, precipitation, and terrain patches

Why do weather forecasters look at **three different maps** instead of just a single satellite photo? Because clouds, falling rain, and ground terrain each tell a different part of the story.

When you run [`fetch_weather_patch.py`](fetch_weather_patch.py) below, you pull a `64 x 64` pixel patch around the storm center and render three side-by-side panels:

1. **Panel 1 — Cloud & Moisture (`NOAA/GOES/19/MCMIPF` or `NOAA/GOES/16/MCMIPF`)**:
   * **What it is**: Geostationary weather satellites orbit 35,786 km above the equator at the exact speed of Earth's rotation, snapping a continuous image of the Western Hemisphere every 10 minutes (**GOES-19** replaced **GOES-16** as the operational GOES-East satellite on April 7, 2025).
   * **What you are looking at**: Bright white and cyan swirls show towering, ice-cold cloud tops and deep tropical moisture spiraling into the storm's eyewall; darker areas indicate thinner clouds or clearer air.
2. **Panel 2 — Surface Rainfall Rate (`NASA/GPM_L3/IMERG_V07`, in `mm/hr`)**:
   * **What it is**: High clouds do not always produce heavy rain at the ground. NASA's Global Precipitation Measurement (**GPM IMERG V07**) constellation uses spaceborne radar and passive microwave sensors to peer *through* the cloud tops and measure actual falling rain in millimeters per hour (`mm/hr`).
   * **How to read the colors**: Dark navy is dry (`0 mm/hr`), cyan/green is light-to-moderate rain (`2–10 mm/hr`), yellow/amber is a heavy downpour (`15–25 mm/hr`), and crimson/magenta (`>30 mm/hr`) indicates torrential rainfall capable of triggering rapid flash flooding.
3. **Panel 3 — Terrain Elevation (`MERIT/DEM/v1_0_3`, in meters)**:
   * **What it is**: A high-precision Digital Elevation Model (**DEM**) showing the height of the land surface above sea level (dark blue = ocean at `0 m`, green = coastal plains, brown/tan = steep mountain ridges).
   * **Why mountains matter for rain (*orographic lift*)**: Mountains act like physical ramps. When moist hurricane winds slam into a steep range—like Jamaica's Blue Mountains (`2,256 m`) or North Carolina's Blue Ridge (`2,037 m`)—the air is forced rapidly upward, cools, and squeezes out far more rain on the windward slopes than over flat ground.

In [ ]:
import fetch_weather_patch
import visualize

patch_data = fetch_weather_patch.get_storm_patch_arrays(
    project_id=PROJECT_ID,
    storm_preset=STORM_PRESET,
    patch_size=64,
)
print(f"Storm:       {patch_data['storm_name']}")
print(f"Timestamp:   {patch_data['iso_time']}")
print(f"Satellite:   {patch_data['goes_satellite']}")
print(f"Data Source: {patch_data['source']}")

overview_fig = visualize.create_storm_overview_figure(patch_data)
overview_fig.show()

Save the 3-panel visualization as a single stitched composite PNG image (`storm_composite.png`) and display it inline below. In Step 4, you pass this exact PNG image to **Gemini 3.8 Flash** (`types.Part.from_bytes`) so the model can visually inspect where the heaviest rainbands (center panel) overlap with steep mountain terrain (right panel).

In [ ]:
from IPython.display import Image as IPyImage, display

COMPOSITE_IMAGE_PATH = "storm_composite.png"
png_bytes = visualize.export_composite_png_bytes(patch_data)
with open(COMPOSITE_IMAGE_PATH, "wb") as f:
    f.write(png_bytes)
print(f"Saved {COMPOSITE_IMAGE_PATH} ({len(png_bytes):,} bytes) — exact image sent to Gemini:")
display(IPyImage(filename=COMPOSITE_IMAGE_PATH))

## 2. Load hourly storm observations into BigQuery with `Apache Beam`

A single satellite snapshot is like a still photograph—it shows you where the storm is at one moment, but not whether rainfall is accelerating or tapering off.

To forecast upcoming rain, you need a chronological **time series** (hourly measurements over time). In [`load_weather_time_series.py`](load_weather_time_series.py) (`[START dataflow_weather_load_time_series]`), you run an **Apache Beam** pipeline that transforms 36 consecutive hours of storm observations (`observation_time`, `storm_name`, `precipitation_mm_hr`, `cloud_moisture_index`, and `elevation_m`) and loads them into a BigQuery table.

* **What to look for in the output**: The script creates the BigQuery dataset if needed, runs the Apache Beam pipeline (~15–25 seconds on `DirectRunner`), and prints the destination `Resource ID`, `Loaded Hours: 36`, and the storm's peak observed rainfall (`51.80 mm/hr`).

In [ ]:
# Expected duration: ~15-25 seconds on DirectRunner
!python load_weather_time_series.py \
  --project={PROJECT_ID} \
  --dataset={BIGQUERY_DATASET} \
  --table={BIGQUERY_TABLE} \
  --runner=DirectRunner \
  --temp_location=gs://{BUCKET_NAME}/temp

## 3. Zero-shot precipitation nowcasting with BigQuery ML `AI.FORECAST` (`TimesFM 2.5`)

Once your hourly storm history is in BigQuery, how do you predict the next 6 hours of rainfall?

Previously, you had to export thousands of data patches, rent GPUs, and train a custom neural network for hours. Today, BigQuery ML provides **zero-shot forecasting** via `AI.FORECAST` powered by Google Research's **TimesFM 2.5** foundation model. "Zero-shot" means you do not train a model at all—TimesFM has already learned complex temporal patterns (like storm build-up and decay) from billions of time-series points, so you call it directly in SQL.

* **What to look for in the output**: For each of the next 6 hours (`horizon => 6`), `forecast_precipitation.py` prints the central forecast (`mm/hr`) alongside a **90% confidence interval (`90% CI: [low, high]`)**.

In [ ]:
# Expected duration: ~10-20 seconds
!python forecast_precipitation.py {PROJECT_ID} {BIGQUERY_DATASET} {BIGQUERY_TABLE}

### Visualize observed `GPM IMERG V07` rainfall versus `AI.FORECAST` nowcast

Run the cell below to plot the 36 hours of observed satellite rainfall alongside the 6-hour `AI.FORECAST` prediction:

* **Blue solid line (`Observed GPM IMERG V07 Rainfall`)**: Shows the storm ramping up from outer rainbands (`~2 mm/hr`) to eyewall landfall (`51.8 mm/hr`) and beginning to weaken after crossing the mountain crest.
* **Red dashed line (`BigQuery AI.FORECAST Nowcast`)**: Shows TimesFM 2.5's central hourly forecast for the next 6 hours.
* **Shaded coral fan (`90% Confidence Interval`)**: Shows the range between the 5th and 95th percentile predictions. Hydrologists and emergency managers closely watch the **top edge** of this shaded band—because if the storm stalls over the mountains for even two extra hours, rainfall tracks the upper bound.

In [ ]:
from google.cloud import bigquery

bq_client = bigquery.Client(project=PROJECT_ID)
table_ref = f"{PROJECT_ID}.{BIGQUERY_DATASET}.{BIGQUERY_TABLE}"

hist_sql = f"SELECT observation_time, precipitation_mm_hr FROM `{table_ref}` ORDER BY observation_time ASC"
fc_sql = f"""
    SELECT
      forecast_timestamp,
      forecast_value AS forecasted_precipitation_mm_hr,
      prediction_interval_lower_bound AS lower_bound_mm_hr,
      prediction_interval_upper_bound AS upper_bound_mm_hr
    FROM
      AI.FORECAST(
        TABLE `{table_ref}`,
        data_col => 'precipitation_mm_hr',
        timestamp_col => 'observation_time',
        model => 'TimesFM 2.5',
        id_cols => ['storm_name'],
        horizon => 6,
        confidence_level => 0.90
      )
    ORDER BY forecast_timestamp ASC
"""
historical_rows = [dict(r) for r in bq_client.query(hist_sql).result()]
forecast_rows = [dict(r) for r in bq_client.query(fc_sql).result()]

forecast_fig = visualize.create_forecast_comparison_figure(
    historical_rows=historical_rows,
    forecast_rows=forecast_rows,
    storm_name="Hurricane Melissa",
)
forecast_fig.show()

## 4. Multimodal storm and orographic flood-risk synthesis with Gemini 3.8 Flash

Numbers in a database (`38.4 mm/hr`) only tell half the story—impact depends heavily on **geography**. Thirty millimeters of rain per hour over flat ocean is harmless, whereas the same rain falling onto already-saturated `2,000 m` mountain slopes funnels into narrow river gorges and triggers catastrophic flash floods and mudslides.

In [`analyze_storm_risk.py`](analyze_storm_risk.py) (`[START googlegenaisdk_weather_analyze_storm_risk]`), you pass **both** the 3-panel satellite/terrain image (`storm_composite.png`) and the `AI.FORECAST` hourly trajectory to **Gemini 3.8 Flash** (`gemini-3.8-flash` on the `global` endpoint) using a Pydantic `StormRiskAssessment` schema:

* **`Display Name` (`flood_risk_level`)**: Overall flood severity classification.
* **`Orographic Amplification Risk`**: Plain-language explanation of how the terrain in Panel 3 is interacting with the rainbands in Panel 2.
* **`Storm Summary` & bulleted actions**: Concrete, prioritized emergency response steps.

In [ ]:
os.environ.pop("WEATHER_FORECAST_CONTEXT", None)
!python analyze_storm_risk.py {PROJECT_ID} {COMPOSITE_IMAGE_PATH}

## 5. Customize for local current weather forecasting (`San Francisco Bay Area`)

How do you adapt this exact workflow from a historic hurricane to **today's live weather in your own backyard**?

By default, the cell below is configured for the **San Francisco Bay Area (`37.7749°N, -122.4194°W`)**, a region famous for sharp **microclimates** where cold Pacific water, coastal fog (the marine layer), and `850 m` coastal mountain ranges create dramatically different weather just a few miles apart:

1. **Automatic West Coast Satellite Selection (`NOAA/GOES/18/MCMIPF`)**: Because the West Coast sits west of `-105°W` longitude, the code automatically switches from `GOES-19` (East) to **GOES-18 (West)** to capture the latest 10-minute cloud and coastal fog scan over California.
2. **Bay Area Topography (`MERIT/DEM/v1_0_3`)**: In Panel 3 of the plot, you can clearly see the flat blue sea-level depression of **San Francisco Bay** (`-2.6 m` to `0 m`) framed by the **Marin Headlands / Mount Tamalpais** to the north, the **Santa Cruz Mountains** (`851 m`) to the south, and the **East Bay Hills / Diablo Range** to the east.
3. **Live 72-Hour Forecast (`NOAA/GFS0P25`)**: Queries the latest run of NOAA's Global Forecast System (`0.25°` resolution) in Earth Engine for the next 72 hours (`+6h` to `+72h`), printing temperature (`°C` and `°F`), relative humidity (`RH %`), wind speed (`m/s`), cloud cover (`%`), and precipitation (`mm`).

> **Try customizing it**: Change `LOCAL_REGION_NAME`, `LOCAL_LATITUDE`, and `LOCAL_LONGITUDE` in the form below to any city or watershed to pull its live satellite view, terrain map, and 72-hour forecast.

In [ ]:
LOCAL_REGION_NAME = "San Francisco Bay Area, CA"  # @param {type:"string"}
LOCAL_LATITUDE = 37.7749  # @param {type:"number"}
LOCAL_LONGITUDE = -122.4194  # @param {type:"number"}
LOCAL_REGION_NAME = os.environ.get("LOCAL_REGION_NAME", LOCAL_REGION_NAME)

local_result = fetch_weather_patch.get_local_weather_forecast(
    project_id=PROJECT_ID,
    region_name=LOCAL_REGION_NAME,
    latitude=float(LOCAL_LATITUDE),
    longitude=float(LOCAL_LONGITUDE),
)
local_patch = local_result["patch_data"]
print(f"Region:      {local_patch['storm_name']}")
print(f"Satellite:   {local_patch['goes_satellite']}")
print(f"Data Source: {local_patch['source']}")
for row in local_result["forecast_rows"]:
    print(
        f"  +{row['lead_hours']:02d}h ({row['forecast_time_utc']}): "
        f"{row['temp_c']:.1f}°C ({row['temp_f']:.1f}°F), "
        f"RH {row['relative_humidity_pct']:.0f}%, wind {row['wind_speed_ms']:.1f} m/s, "
        f"clouds {row['cloud_cover_pct']:.0f}%, precip {row['precip_mm']:.2f} mm"
    )

local_fig = visualize.create_storm_overview_figure(local_patch)
local_fig.show()

### Synthesize a localized microclimate and weather risk report with Gemini 3.8 Flash

Save and display your live local 3-panel composite image (`local_weather_composite.png`), then pass it alongside the 72-hour `NOAA/GFS0P25` forecast table into [`analyze_storm_risk.py`](analyze_storm_risk.py). Notice how `gemini-3.8-flash` adapts its assessment—distinguishing benign coastal marine-layer clouds or light drizzle over Bay Area hills from severe cyclonic flooding.

In [ ]:
from IPython.display import Image as IPyImage, display

LOCAL_COMPOSITE_PATH = "local_weather_composite.png"
with open(LOCAL_COMPOSITE_PATH, "wb") as f:
    f.write(visualize.export_composite_png_bytes(local_patch))
print(f"Saved {LOCAL_COMPOSITE_PATH} — exact image sent to Gemini:")
display(IPyImage(filename=LOCAL_COMPOSITE_PATH))

os.environ["WEATHER_FORECAST_CONTEXT"] = local_result["forecast_context"]
!python analyze_storm_risk.py {PROJECT_ID} {LOCAL_COMPOSITE_PATH}

## 6. Scaling to global medium-range ensemble forecasts with `WeatherNext`

Why do weather apps say *"30% chance of heavy rain in 4 days"* instead of giving a single guaranteed number? Because small ripples in today's atmosphere can grow into large differences several days out.

To capture that uncertainty, **Google DeepMind's WeatherNext 3** (`projects/gcp-public-data-weathernext/assets/weathernext_3_0_0_0p1deg`) does not run just one simulation—it runs **64 slightly different simulations ("ensemble members") every single hour**. In Google Earth Engine, all 64 simulations are pre-summarized into **percentile bands** across 114 atmospheric variables:

* **`p50` (Ensemble Median — dark blue line)**: The middle scenario where half of the 64 simulations predict less rain and half predict more.
* **`p25` to `p75` (Interquartile Range — darker blue band)**: The middle 50% of all 64 simulations—the "most likely" corridor.
* **`p10` to `p90` (10th to 90th Percentile — outer light blue band & orange dotted line)**: Covers 80% of all 64 members. When the **`p90` tail-risk line** spikes much higher than `p50`, it alerts forecasters days in advance that a subset of storm tracks could produce extreme rainfall.

Run the executable cell below to query the live `weathernext_3_0_0_0p1deg` collection in Earth Engine, inspect the latest hourly initialization metadata and ensemble band structure, and render an interactive **64-member WeatherNext 3 ensemble plume chart**.

In [ ]:
wn_data = fetch_weather_patch.get_weathernext_ensemble_forecast(
    project_id=PROJECT_ID,
    region_name="Hurricane Melissa (Jamaica Landfall)",
    latitude=18.1,
    longitude=-78.0,
)
print(f"Collection ID:        {wn_data['collection_id']}")
print(f"Latest Live Asset ID: {wn_data['latest_asset_id']}")
print(f"Latest Init Time:     {wn_data['latest_init_utc']}")
print(f"Total Ensemble Bands: {wn_data['total_bands']}")
print(f"Precip Percentiles:   {', '.join(wn_data['precip_bands'])}")
for r in wn_data["ensemble_rows"]:
    print(
        f"  +{r['lead_hours']:02d}h ({r['valid_time_utc']}): "
        f"p50={r['p50_mm_hr']:4.1f} mm/hr | mean={r['mean_mm_hr']:4.1f} mm/hr | "
        f"p25-p75=[{r['p25_mm_hr']:4.1f}, {r['p75_mm_hr']:4.1f}] | "
        f"p10-p90=[{r['p10_mm_hr']:4.1f}, {r['p90_mm_hr']:4.1f}]"
    )

wn_fig = visualize.create_weathernext_ensemble_figure(wn_data)
wn_fig.show()

## Cleaning up

To avoid incurring ongoing charges to your Google Cloud account for the resources created in this tutorial, delete the BigQuery dataset and temporary Cloud Storage staging files.

In [ ]:
!bq rm -r -f -d {PROJECT_ID}:{BIGQUERY_DATASET}
!gcloud storage rm --recursive gs://{BUCKET_NAME}/temp